# Sailing Labeling

Extracts frames from sailing videos and saves them with their labels.

| Step | What happens |
|---|---|
| 0. Setup | Install packages, imports, mount Drive, constants |
| 1. Load jobs | Read video list from Excel, validate |
| 2. Extract frames | Sample frames, resize, save session folders and label CSVs |
| 3. Scan and split | Build train/val/test manifests from all sessions on disk |
| 4. Class info | Save class_info.json and distribution plots |

**FPS rule:**
- `no-go zone` videos are extracted at **15 FPS** — the direction is hardest to classify and benefits from more frames
- All other directions are extracted at **5 FPS** — sufficient for stable classes


## 0. Setup

In [37]:
# Install required packages if not already present
import subprocess, sys, importlib

packages = {
    'cv2':       'opencv-python-headless',
    'tqdm':      'tqdm',
    'matplotlib':'matplotlib',
    'sklearn':   'scikit-learn',
    'pandas':    'pandas',
    'openpyxl':  'openpyxl',
}

for module_name, pip_name in packages.items():
    try:
        importlib.import_module(module_name)
        print(f'{pip_name} already installed')
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', pip_name, '--quiet'],
                       capture_output=True)
        print(f'{pip_name} installed')

print('All dependencies ready')


opencv-python-headless already installed
tqdm already installed
matplotlib already installed
scikit-learn already installed
pandas already installed
openpyxl already installed
All dependencies ready


In [38]:
import os
import json
import csv
import shutil
import time
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import openpyxl
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import StratifiedShuffleSplit

print('Imports OK')


Imports OK


In [39]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT     = Path('/content/drive/MyDrive')
PROJECT_ROOT   = DRIVE_ROOT / 'DeepLearning_sailing'
RAW_VIDEO_ZIP  = PROJECT_ROOT / 'raw video.zip'
RAW_VIDEO_PATH = PROJECT_ROOT / 'raw video'
VIDEO_ROOT     = PROJECT_ROOT / 'video'
DATA_DIR       = PROJECT_ROOT / 'data'
RESULTS_DIR    = PROJECT_ROOT / 'results'
TMP_DIR        = Path('/tmp/sailing_pipeline')

for folder in [RAW_VIDEO_PATH, VIDEO_ROOT, DATA_DIR, RESULTS_DIR, TMP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Drive mounted')
print(f'Project root: {PROJECT_ROOT}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted
Project root: /content/drive/MyDrive/DeepLearning_sailing


In [40]:
print(RAW_VIDEO_ZIP.exists(), RAW_VIDEO_ZIP.stat().st_size if RAW_VIDEO_ZIP.exists() else None)

True 1960478442


### Local VM extraction

Zip files are stored on Drive and extracted once per session to the local VM filesystem.
Local reads are 10-20x faster than Drive reads — this makes frame extraction and
dataset scanning significantly faster.

Re-running this cell is safe: extraction is skipped if files are already present.


In [41]:
import zipfile, os
from tqdm import tqdm

# Extract a zip from Drive to the local VM filesystem.
# Skips extraction if the target directory already contains files (i.e. extraction was already done this session).
# Shows a tqdm progress bar over the zip entries.

def extract_zip(zip_drive_path, extract_to, label):
    zip_drive_path = Path(zip_drive_path)
    extract_to     = Path(extract_to)

    if not zip_drive_path.exists():
        raise FileNotFoundError(f'{label} zip not found: {zip_drive_path}')

    # Skip if already extracted this session
    existing = list(extract_to.rglob('*')) if extract_to.exists() else []
    if existing:
        print(f'{label}: already extracted ({len(existing)} entries found in {extract_to}), skipping.')
        return

    extract_to.mkdir(parents=True, exist_ok=True)
    zip_size = zip_drive_path.stat().st_size / 1e6
    print(f'Extracting {label} ({zip_size:.0f} MB) -> {extract_to} ...')

    with zipfile.ZipFile(zip_drive_path, 'r') as zf:
        members = zf.infolist()
        for member in tqdm(members, desc=label, unit='file'):
            zf.extract(member, extract_to)

    print(f'{label}: done ({len(members)} files extracted)')


# Zip paths on Drive — adjust filenames if needed
RAW_VIDEO_ZIP = PROJECT_ROOT / 'raw video.zip'

# Local VM extraction targets
LOCAL_RAW_VIDEO = Path('/content/raw_video')

extract_zip(RAW_VIDEO_ZIP, LOCAL_RAW_VIDEO, 'raw_video')

# ── Trova la root effettiva dei video dentro l'unzippato ──────────────────────
# Lo zip potrebbe contenere una sottocartella (es. "raw video/")
# invece di mettere i file direttamente nella root.
def find_video_root(base_dir, video_exts):
    """Risale alle directory finché non trova file video."""
    base_dir = Path(base_dir)
    # Cerca tutti i file video ricorsivamente
    found = []
    for ext in video_exts:
        found.extend(base_dir.rglob(f'*{ext}'))
        found.extend(base_dir.rglob(f'*{ext.upper()}'))
    if not found:
        raise RuntimeError(f'Nessun video trovato sotto {base_dir}')
    # La root è la directory comune più profonda che li contiene tutti
    # (nella pratica: la cartella del primo video trovato, se sono tutti piatti)
    candidate = found[0].parent
    print(f'Video trovati: {len(found)} | Root rilevata: {candidate}')
    return candidate

VIDEO_EXTS = {'.mp4', '.mov', '.avi', '.mkv', '.m4v', '.wmv'}
RAW_VIDEO_PATH = find_video_root(LOCAL_RAW_VIDEO, VIDEO_EXTS)
print(f'RAW_VIDEO_PATH -> {RAW_VIDEO_PATH}')


raw_video: already extracted (279 entries found in /content/raw_video), skipping.
Video trovati: 272 | Root rilevata: /content/raw_video/raw video
RAW_VIDEO_PATH -> /content/raw_video/raw video


In [42]:
# Class labels
# Defining labels
DIRECTION_CLASSES = ['no-go zone', 'close-hauled', 'close reach', 'beam reach', 'broad reach', 'run']
SIDE_CLASSES      = ['port', 'starboard']
BOAT_TYPE_CLASSES = ['ILCA', 'Optimist']

# Creating dictionaries of direction and side classes
DIR2IDX  = {name: idx for idx, name in enumerate(DIRECTION_CLASSES)}
SIDE2IDX = {name: idx for idx, name in enumerate(SIDE_CLASSES)}

# Dictionaries useful for inference
IDX2DIR  = {idx: name for name, idx in DIR2IDX.items()}
IDX2SIDE = {idx: name for name, idx in SIDE2IDX.items()}

# FPS rule:
# no-go zone is the hardest class to detect (very narrow wind window) so we extract more frames per second to give the model more examples of it.
# All other directions are extracted at a lower rate to keep the dataset balanced.

FPS_NO_GO_ZONE = 15    # frames per second for no-go zone videos
FPS_DEFAULT    = 5     # frames per second for all other directions

IMG_SIZE    = 224      # every frame is resized to this square size

# Video format accepted
VIDEO_EXTS  = {'.mp4', '.mov', '.avi', '.mkv', '.m4v', '.wmv'}

# Train / val / test split ratios
TRAIN_RATIO = 0.65
VAL_RATIO   = 0.20
TEST_RATIO  = 0.15
RANDOM_SEED = 42

# Setting seeds
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('Constants defined')
print(f'Direction classes ({len(DIRECTION_CLASSES)}): {DIRECTION_CLASSES}')
print(f'Side classes      ({len(SIDE_CLASSES)}): {SIDE_CLASSES}')
print(f'Boat type classes : {BOAT_TYPE_CLASSES}')
print(f'FPS no-go zone    : {FPS_NO_GO_ZONE}')
print(f'FPS default       : {FPS_DEFAULT}')


Constants defined
Direction classes (6): ['no-go zone', 'close-hauled', 'close reach', 'beam reach', 'broad reach', 'run']
Side classes      (2): ['port', 'starboard']
Boat type classes : ['ILCA', 'Optimist']
FPS no-go zone    : 15
FPS default       : 5


## 1. Load Jobs from Excel

`video_jobs.xlsx` has one row per video with columns: `video`, `direction`, `side`, `boat_type`, `processed`.
Videos with `processed = 1` are skipped.


In [43]:
# Excel file path — rimane su Drive
JOBS_EXCEL = PROJECT_ROOT / 'video_jobs.xlsx'

if not JOBS_EXCEL.exists():
    raise FileNotFoundError(f'video_jobs.xlsx not found at {JOBS_EXCEL}')

# Google Drive sometimes needs a moment to sync a file after mounting
def wait_for_file(path, retries=5, delay=3):
    for attempt in range(1, retries + 1):
        try:
            with open(path, 'rb') as f:
                data = f.read()
            if len(data) > 0:
                print(f'{path.name} ready ({len(data):,} bytes)')
                return
        except OSError:
            pass
        print(f'Waiting for Drive sync, attempt {attempt}/{retries}...')
        time.sleep(delay)
    raise RuntimeError(f'Could not read {path} after {retries} attempts')

# Return the column number of a named column in the header row
def find_col(header_row, col_name):
    for cell in header_row:
        if cell.value and str(cell.value).strip().lower() == col_name:
            return cell.column
    raise ValueError(f'Column "{col_name}" not found in header row')

wait_for_file(JOBS_EXCEL)

# Open the file in read mode on the page called jobs
wb = openpyxl.load_workbook(JOBS_EXCEL, read_only=True)
ws = wb['jobs']

# Find the row that contains the word 'video' as the header row
header_row = None
for row in ws.iter_rows():
    if any(str(cell.value).strip().lower() == 'video' for cell in row if cell.value):
        header_row = row
        break

if header_row is None:
    raise RuntimeError('Could not find the header row in video_jobs.xlsx')

# Search each column
col_video     = find_col(header_row, 'video')
col_direction = find_col(header_row, 'direction')
col_side      = find_col(header_row, 'side')
col_boat_type = find_col(header_row, 'boat_type')
col_processed = find_col(header_row, 'processed')
header_row_idx = header_row[0].row

# Read every row that has not been processed yet
JOBS = []
n_total = 0

for row in ws.iter_rows(min_row=header_row_idx + 1):
    video_cell = row[col_video - 1]
    if not video_cell.value or str(video_cell.value).strip() == '':
        continue
    n_total += 1
    processed_val = row[col_processed - 1].value
    if processed_val is not None and int(processed_val) == 1:
        continue
    JOBS.append({
        'video':     str(row[col_video - 1].value).strip(),
        'direction': str(row[col_direction - 1].value).strip(),
        'side':      str(row[col_side - 1].value).strip(),
        'boat_type': str(row[col_boat_type - 1].value).strip(),
    })

wb.close()

skipped = n_total - len(JOBS)
if skipped:
    print(f'Skipping {skipped} already-processed video(s)')

# Build the boat-type vocabulary from job list
seen = set()
BOAT_TYPE_CLASSES = []
for job in JOBS:
    if job['boat_type'] not in seen:
        BOAT_TYPE_CLASSES.append(job['boat_type'])
        seen.add(job['boat_type'])

BOAT2IDX = {name: idx for idx, name in enumerate(BOAT_TYPE_CLASSES)}
IDX2BOAT = {idx: name for name, idx in BOAT2IDX.items()}

# Validate every job — Excel su Drive, video nell'unzippato locale
errors = []
for i, job in enumerate(JOBS):
    if job['direction'] not in DIRECTION_CLASSES:
        errors.append(f'Job {i} ({job["video"]}): unknown direction "{job["direction"]}"')
    if job['side'] not in SIDE_CLASSES:
        errors.append(f'Job {i} ({job["video"]}): unknown side "{job["side"]}"')

    # Ricerca ricorsiva: funziona qualunque sia la struttura interna dello zip
    matches = list(RAW_VIDEO_PATH.rglob(job['video']))
    if not matches:
        errors.append(f'Job {i}: file not found -> {job["video"]} (cercato sotto {RAW_VIDEO_PATH})')
    else:
        job['_full_path'] = matches[0]  # path completo risolto, usato dall'estrazione frames
        if len(matches) > 1:
            print(f'  WARN: job {i} ({job["video"]}): trovati {len(matches)} match, uso il primo: {matches[0]}')

if errors:
    for e in errors:
        print(f'  ERROR: {e}')
    raise ValueError(f'{len(errors)} error(s) found, fix them before continuing')

print(f'{len(JOBS)} job(s) loaded and validated')
print(f'Boat types: {BOAT_TYPE_CLASSES}')
print()
for job in JOBS:
    fps = FPS_NO_GO_ZONE if job['direction'] == 'no-go zone' else FPS_DEFAULT
    print(f'  {job["video"]:45s}  {job["direction"]:15s}  {job["side"]:10s}  {job["boat_type"]:10s}  {fps} fps')

video_jobs.xlsx ready (63,114 bytes)
Skipping 132 already-processed video(s)
0 job(s) loaded and validated
Boat types: []



## 2. Frame Extraction

For each video:
1. Determine the target FPS: 15 for no-go zone, 5 for everything else
2. Extract frames using ffmpeg (OpenCV as fallback)
3. Resize each frame to `IMG_SIZE x IMG_SIZE`
4. Save frames to a session folder
5. Write a `_labels.csv` with filename, labels, and timestamps


In [45]:
 # Read the native FPS of the source video using OpenCV
def get_video_fps(video_path):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    cap.release()
    return fps

# Use ffmpeg to sample exactly fps frames per second from the video.
# Returns a list of (saved_path, original_frame_index_in_source) pairs.
def extract_frames_ffmpeg(video_path, out_dir, fps):
    out_dir.mkdir(parents=True, exist_ok=True)
    src_fps = get_video_fps(video_path)

    cmd = [
        'ffmpeg',
        '-y',                             # Override
        '-i',
        str(video_path),
        '-vf',
        f'fps={fps}',                     # Keep exactly fps frames per second
        '-vsync',
        'vfr',                            # Variable frame rate: no duplicate or dropped frames
        '-q:v',
        '2',                              # high quality JPEG output
        str(out_dir / 'frame_%06d.jpg'),  # File name
        '-hide_banner',
        '-loglevel', 'error'              # Soppress all ecxept errors
    ]
    result = subprocess.run(cmd, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(f'ffmpeg failed: {result.stderr.decode()}')

    paths = sorted(out_dir.glob('frame_*.jpg'))
    results = []
    # Map the sequential output index back to the corresponding frame in the source video
    for seq_idx, p in enumerate(paths):
        orig_frame_idx = round(seq_idx * src_fps / fps)
        results.append((p, orig_frame_idx))
    return results

# OpenCV fallback: seek directly to each target frame to avoid reading every frame.
# Returns a list of (saved_path, original_frame_index_in_source) pairs.
def extract_frames_cv2(video_path, out_dir, fps):
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    src_fps     = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    results = []
    saved   = 0
    out_idx = 0    # index in the downsampled output sequence

    while True:
        target = round(out_idx * src_fps / fps)   # source frame to jump to
        if target >= total_frames:
            break
        cap.set(cv2.CAP_PROP_POS_FRAMES, target)  # seek to that frame
        ret, frame = cap.read()
        if not ret:
            break
        p = out_dir / f'frame_{saved:06d}.jpg'
        cv2.imwrite(str(p), frame)
        results.append((p, target))
        saved   += 1
        out_idx += 1

    cap.release()
    return results

# Resize a BGR frame to size x size pixels using area interpolation
def resize_frame(frame_bgr, size=IMG_SIZE):
    return cv2.resize(frame_bgr, (size, size), interpolation=cv2.INTER_AREA)

# Defining the FPS target based on the job
def target_fps_for_job(job):
    if job['direction'] == 'no-go zone':
        return FPS_NO_GO_ZONE
    return FPS_DEFAULT

# Extract all frames for one video job and save them with a labels CSV.
# Returns a summary dict with frame count and timing info.
def process_job(job):
    video_path   = RAW_VIDEO_PATH / job['video']
    stem         = video_path.stem
    src_fps      = get_video_fps(video_path)
    fps          = target_fps_for_job(job)        # per-job FPS decision
    session_name = f"{stem}_{job['boat_type']}_{job['direction']}_{job['side']}"
    session_dir  = VIDEO_ROOT / session_name
    session_dir.mkdir(parents=True, exist_ok=True)

    tmp_dir = TMP_DIR / session_name
    try:
        frame_pairs = extract_frames_ffmpeg(video_path, tmp_dir, fps)
        # Fallback function useful in case of faliure ffmpeg
    except Exception:
        frame_pairs = extract_frames_cv2(video_path, tmp_dir, fps)

    csv_columns = [
        'filename', 'frame_number', 'orig_frame_idx', 'timestamp_s',
        'video_stem', 'direction', 'side', 'boat_type',
        'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2', 'boat_detected',
    ]

    label_rows = []
    # for each frame extracted in frame_pairs
    for frame_number, (raw_path, orig_frame_idx) in enumerate(
            tqdm(frame_pairs, desc=session_name, leave=False)):

        frame_bgr = cv2.imread(str(raw_path))      # Reads the jpeg file
        if frame_bgr is None:
            continue

        resized = resize_frame(frame_bgr)           # resize to IMG_SIZE x IMG_SIZE
        fname   = raw_path.name                     # Rename the file
        cv2.imwrite(str(session_dir / fname), resized)

        # Timestamp in seconds: maps the source frame index back to real time
        timestamp_s = round(orig_frame_idx / src_fps, 4)

        label_rows.append({
            'filename':       fname,
            'frame_number':   frame_number,
            'orig_frame_idx': orig_frame_idx,
            'timestamp_s':    timestamp_s,
            'video_stem':     stem,
            'direction':      job['direction'],
            'side':           job['side'],
            'boat_type':      job['boat_type'],
            'bbox_x1': 0, 'bbox_y1': 0,
            'bbox_x2': IMG_SIZE, 'bbox_y2': IMG_SIZE,
            'boat_detected': False,
        })
    # Updating csv with all informations
    csv_path = session_dir / f'{session_name}_labels.csv'
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=csv_columns)
        writer.writeheader()
        writer.writerows(label_rows)

    shutil.rmtree(tmp_dir, ignore_errors=True)   # clean up temporary files

    return {
        'session':    session_name,
        'n_frames':   len(label_rows),
        'fps_used':   fps,
        'direction':  job['direction'],
        'side':       job['side'],
        'boat_type':  job['boat_type'],
        'video_stem': stem,
        'duration_s': round(label_rows[-1]['timestamp_s'], 1) if label_rows else 0,
    }


print('Extraction functions ready')
print(f'FPS no-go zone : {FPS_NO_GO_ZONE}')
print(f'FPS default    : {FPS_DEFAULT}')


Extraction functions ready
FPS no-go zone : 15
FPS default    : 5


In [46]:
# Run all jobs
summaries = []

for i, job in enumerate(JOBS, 1):
    fps = target_fps_for_job(job)
    print(f'\n{i}/{len(JOBS)}  {job["video"]}  ({job["direction"]} -> {fps} fps)')
    try:
        summary = process_job(job)
        summary['source_video'] = job['video']
        summaries.append(summary)
        print(f'  {summary["n_frames"]} frames saved -> {summary["session"]}')
        print(f'  Duration: {summary["duration_s"]}s  |  FPS used: {summary["fps_used"]}')
    except Exception as e:
        print(f'  Failed: {e}')

print(f'\nDone. {len(summaries)}/{len(JOBS)} jobs completed.')
print()
for s in summaries:
    print(f'  {s["session"]:55s}  {s["n_frames"]:5d} frames  {s["duration_s"]:6.1f}s  {s["fps_used"]} fps')



Done. 0/0 jobs completed.



In [47]:
# Mark completed jobs as processed=1 in the Excel file so they are skipped next run
completed_videos = {str(s['source_video']).strip() for s in summaries}

wait_for_file(JOBS_EXCEL)
wb_write = openpyxl.load_workbook(JOBS_EXCEL)
ws_write = wb_write['jobs']

# Find the header row again in the writable workbook
header_row_w = None
for row in ws_write.iter_rows():
    if any(str(cell.value).strip().lower() == 'video' for cell in row if cell.value):
        header_row_w = row
        break

col_video_w     = find_col(header_row_w, 'video')
col_processed_w = find_col(header_row_w, 'processed')
header_idx_w    = header_row_w[0].row

updated = 0
for row in ws_write.iter_rows(min_row=header_idx_w + 1):
    video_cell = row[col_video_w - 1]
    proc_cell  = row[col_processed_w - 1]
    if video_cell.value and str(video_cell.value).strip() in completed_videos:
        proc_cell.value = 1
        updated += 1

wb_write.save(JOBS_EXCEL)
print(f'video_jobs.xlsx updated: {updated} video(s) marked as processed=1')


video_jobs.xlsx ready (63,114 bytes)
video_jobs.xlsx updated: 0 video(s) marked as processed=1


## 3. Scan All Sessions and Split into Train / Val / Test

We scan all session folders on disk (including those from previous runs) so that
re-running the notebook with new videos merges them into the existing dataset.
The split is done at the session level so the same source video never appears
in two different splits.


In [49]:
def scan_all_sessions(root):
    # Walk every session folder under root and collect frame records into a DataFrame
    records = []
    all_dirs = sorted(d for d in root.iterdir() if d.is_dir())
    for session_dir in tqdm(all_dirs, desc='Scanning sessions', unit='session'):
        if not session_dir.is_dir():
            continue
        csv_files = list(session_dir.glob('*_labels.csv'))
        if not csv_files:
            continue
        with open(csv_files[0], newline='') as f:
            for row in csv.DictReader(f):
                direction  = row['direction'].strip()
                side       = row['side'].strip()
                boat_type  = row.get('boat_type', '').strip()
                frame_path = session_dir / row['filename']

                if direction not in DIR2IDX or side not in SIDE2IDX:
                    continue
                if not boat_type or not frame_path.exists():
                    continue

                records.append({
                    'abs_path':       str(frame_path),
                    'frame_number':   int(row.get('frame_number', 0)),
                    'orig_frame_idx': int(row.get('orig_frame_idx', 0)),
                    'timestamp_s':    float(row.get('timestamp_s', 0.0)),
                    'video_stem':     row.get('video_stem', session_dir.name),
                    'direction':      direction,
                    'side':           side,
                    'boat_type':      boat_type,
                    'dir_idx':        DIR2IDX[direction],
                    'side_idx':       SIDE2IDX[side],
                    'session':        session_dir.name,
                    'bbox_x1':        int(row.get('bbox_x1', 0)),
                    'bbox_y1':        int(row.get('bbox_y1', 0)),
                    'bbox_x2':        int(row.get('bbox_x2', IMG_SIZE)),
                    'bbox_y2':        int(row.get('bbox_y2', IMG_SIZE)),
                    'boat_detected':  row.get('boat_detected', 'False') == 'True',
                })
    return pd.DataFrame(records)


df_all = scan_all_sessions(VIDEO_ROOT)

# Rebuild the boat-type vocabulary from everything on disk
BOAT_TYPE_CLASSES = sorted(df_all['boat_type'].unique().tolist())
BOAT2IDX = {name: idx for idx, name in enumerate(BOAT_TYPE_CLASSES)}
IDX2BOAT = {idx: name for name, idx in BOAT2IDX.items()}
df_all['boat_type_idx'] = df_all['boat_type'].map(BOAT2IDX)

df = df_all

print(f'Found {len(df):,} frames across {df["session"].nunique()} session(s)')
print(f'Videos      : {df["video_stem"].nunique()}')
print(f'Timestamps  : {df["timestamp_s"].min():.1f}s - {df["timestamp_s"].max():.1f}s')
print('\nDirection counts:\n')
print(df['direction'].value_counts().to_string())
print('\nSide counts:\n')
print(df['side'].value_counts().to_string())
print('\nBoat type counts:\n')
print(df['boat_type'].value_counts().to_string())


Scanning sessions: 0session [00:00, ?session/s]


KeyError: 'boat_type'

In [ ]:
df_all = scan_all_sessions(VIDEO_ROOT)

# Diagnose before doing anything with df_all
print(f'VIDEO_ROOT: {VIDEO_ROOT}')
print(f'Subfolders found: {sum(1 for p in VIDEO_ROOT.iterdir() if p.is_dir())}')

if df_all.empty:
    # List what is actually in VIDEO_ROOT so you can see the problem
    print('\nContents of VIDEO_ROOT:')
    for p in sorted(VIDEO_ROOT.iterdir()):
        csv_files = list(p.glob('*_labels.csv')) if p.is_dir() else []
        print(f'  {p.name}  |  csv files: {len(csv_files)}')
        if csv_files:
            # Show the first few column names to check for a naming mismatch
            import csv as _csv
            with open(csv_files[0], newline='') as f:
                reader = _csv.reader(f)
                print(f'    columns: {next(reader)}')
    raise ValueError(
        'scan_all_sessions returned an empty DataFrame.\n'
        'Either VIDEO_ROOT has no session folders, the label CSVs are missing, '
        'or a column is named differently than expected.'
    )

print(f'\nDataFrame shape: {df_all.shape}')
print(f'Columns: {df_all.columns.tolist()}')

In [ ]:
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)


def split_by_session(df, train_frac, val_frac, seed):
    # Split at the session level so no source video appears in two splits.
    # Uses stratified sampling to keep class balance across splits.

    # Creating a unique key for each session given the boat, direction and side
    session_info = (
        df.groupby('session', group_keys=False)
          .apply(lambda g: (g['boat_type'] + '|' + g['direction'] + '|' + g['side']).mode()[0],
                 include_groups=False)
          .reset_index()
    )
    session_info.columns = ['session', 'stratum']

    sessions = session_info['session'].values
    strata   = session_info['stratum'].values

    _, counts = np.unique(strata, return_counts=True)
    if counts.min() < 2:
        raise ValueError('Not enough sessions per class for session-level split')

    test_frac = max(0.0, 1.0 - train_frac - val_frac)

    # Identify test set
    if test_frac > 1e-9:
        splitter1 = StratifiedShuffleSplit(1, test_size=test_frac, random_state=seed)
        tv_idx, te_idx = next(splitter1.split(sessions, strata))
    else:
        tv_idx = np.arange(len(sessions))
        te_idx = np.array([], dtype=int)

    # Select a validation set
    effective_val = val_frac / (train_frac + val_frac)
    splitter2 = StratifiedShuffleSplit(1, test_size=effective_val, random_state=seed)
    tr_idx, va_idx = next(splitter2.split(sessions[tv_idx], strata[tv_idx]))

    return set(sessions[tv_idx][tr_idx]), set(sessions[tv_idx][va_idx]), set(sessions[te_idx])

 # Fallback when there are too few sessions per class for session-level split
def split_by_frame(df, train_frac, val_frac, seed):
    strata   = df['boat_type'] + '|' + df['direction'] + '|' + df['side']
    idx      = np.arange(len(df))
    test_frac = max(0.0, 1.0 - train_frac - val_frac)

    if test_frac > 1e-9:
        s1 = StratifiedShuffleSplit(1, test_size=test_frac, random_state=seed)
        tv_idx, te_idx = next(s1.split(idx, strata))
    else:
        tv_idx = idx
        te_idx = np.array([], dtype=int)

    effective_val = val_frac / (train_frac + val_frac)
    s2 = StratifiedShuffleSplit(1, test_size=effective_val, random_state=seed)
    tr_idx, va_idx = next(s2.split(tv_idx, strata.iloc[tv_idx]))

    return tv_idx[tr_idx], tv_idx[va_idx], te_idx

# Extracting train,val e test set
try:
    tr_sessions, va_sessions, te_sessions = split_by_session(
        df, TRAIN_RATIO, VAL_RATIO, RANDOM_SEED)
    df_train = df[df.session.isin(tr_sessions)].reset_index(drop=True)
    df_val   = df[df.session.isin(va_sessions)].reset_index(drop=True)
    df_test  = df[df.session.isin(te_sessions)].reset_index(drop=True)
    split_mode = 'session_split'
except ValueError:
    tr_i, va_i, te_i = split_by_frame(df, TRAIN_RATIO, VAL_RATIO, RANDOM_SEED)
    df_train = df.iloc[tr_i].reset_index(drop=True)
    df_val   = df.iloc[va_i].reset_index(drop=True)
    df_test  = df.iloc[te_i].reset_index(drop=True)
    split_mode = 'frame_split'

print(f'Split mode: {split_mode}')
print(f'  train={len(df_train):,}  val={len(df_val):,}  test={len(df_test):,} frames')

df_train.to_csv(DATA_DIR / 'train.csv', index=False)
df_val.to_csv(  DATA_DIR / 'val.csv',   index=False)
df_test.to_csv( DATA_DIR / 'test.csv',  index=False)
print('Manifests saved: train.csv, val.csv, test.csv')


## 4. Class Info and Distribution

In [ ]:
# Computing weights for each class based on real video duration (seconds),
# not on extracted frame counts. This avoids the distortion caused by
# no-go zone being extracted at 15 FPS vs 5 FPS for other classes:
# counting frames would make no-go zone look common, suppressing its weight
# and cancelling out the very reason we extracted it at higher FPS.
#
# Formula: duration_s per session = timestamp_s of last frame in that session
# (timestamp_s is the real-time position in the source video, FPS-independent)

def compute_duration_weights(df, label_col, fps_col, classes):
    """
    Inverse-frequency weights based on total video duration per class (seconds),
    computed only on the training sessions in df.
    Uses timestamp_s (real video time) to get duration independent of extraction FPS.
    """
    # Max timestamp_s per session → duration of that session in seconds
    duration_per_session = (
        df.groupby(['session', label_col])['timestamp_s']
        .max()
        .reset_index()
        .rename(columns={'timestamp_s': 'duration_s'})
    )
    # Sum durations across all training sessions per class
    duration_per_class = (
        duration_per_session.groupby(label_col)['duration_s']
        .sum()
    )
    total     = duration_per_class.sum()
    n_present = sum(1 for c in classes if c in duration_per_class.index)
    weights   = {}
    for c in classes:
        if c in duration_per_class.index:
            weights[c] = total / (n_present * duration_per_class[c])
        else:
            weights[c] = 0.0
    return weights, duration_per_class


dir_weights,       dir_durations  = compute_duration_weights(df_train, 'direction', 'fps_used', DIRECTION_CLASSES)
side_weights,      side_durations = compute_duration_weights(df_train, 'side',      'fps_used', SIDE_CLASSES)
boat_type_weights, boat_durations = compute_duration_weights(df_train, 'boat_type', 'fps_used', BOAT_TYPE_CLASSES)

print('Class weights based on real video duration:')
print('\n  Direction:')
for c in DIRECTION_CLASSES:
    d = dir_durations.get(c, 0)
    print(f'    {c:<15}  {d:7.1f}s  →  weight={dir_weights[c]:.3f}')
print('\n  Side:')
for c in SIDE_CLASSES:
    d = side_durations.get(c, 0)
    print(f'    {c:<12}  {d:7.1f}s  →  weight={side_weights[c]:.3f}')
print('\n  Boat type:')
for c in BOAT_TYPE_CLASSES:
    d = boat_durations.get(c, 0)
    print(f'    {c:<10}  {d:7.1f}s  →  weight={boat_type_weights[c]:.3f}')

# Creating a json file with all useful informations for subsequent process
class_info = {
    'direction_classes':  DIRECTION_CLASSES,
    'side_classes':       SIDE_CLASSES,
    'boat_type_classes':  BOAT_TYPE_CLASSES,
    'dir2idx':            DIR2IDX,
    'side2idx':           SIDE2IDX,
    'boat2idx':           BOAT2IDX,
    'dir_weights':        dir_weights,
    'side_weights':       side_weights,
    'boat_type_weights':  boat_type_weights,
    'img_size':           IMG_SIZE,
    # We store both FPS values so the training and inference notebooks know
    # that different classes were extracted at different rates
    'fps_no_go_zone':     FPS_NO_GO_ZONE,
    'fps_default':        FPS_DEFAULT,
    'train_count':        len(df_train),
    'val_count':          len(df_val),
    'test_count':         len(df_test),
    'split_mode':         split_mode,
    'temporal':           True,
}

with open(DATA_DIR / 'class_info.json', 'w') as f:
    json.dump(class_info, f, indent=2)

print('class_info.json saved')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Class Distribution - Full Dataset', fontweight='bold', fontsize=16)

dir_counts = df['direction'].value_counts().reindex(DIRECTION_CLASSES, fill_value=0)
axes[0].bar(dir_counts.index, dir_counts.values, color='steelblue', edgecolor='white')
axes[0].set_title('Direction', fontsize=14)
axes[0].tick_params(axis='x', rotation=30, labelsize=12)
axes[0].tick_params(axis='y', labelsize=12)

side_counts = df['side'].value_counts().reindex(SIDE_CLASSES, fill_value=0)
axes[1].bar(side_counts.index, side_counts.values, color='seagreen', edgecolor='white')
axes[1].set_title('Side', fontsize=14)
axes[1].tick_params(axis='x', labelsize=12)
axes[1].tick_params(axis='y', labelsize=12)

boat_counts = df['boat_type'].value_counts().reindex(BOAT_TYPE_CLASSES, fill_value=0)
axes[2].bar(boat_counts.index, boat_counts.values, color='darkorange', edgecolor='white')
axes[2].set_title('Boat Type', fontsize=14)
axes[2].tick_params(axis='x', rotation=30, labelsize=12)
axes[2].tick_params(axis='y', labelsize=12)

plt.tight_layout()
plt.savefig(DATA_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Font sizes
FS_TITLE    = 17   # suptitle
FS_SUBTITLE = 15   # titoli subplot
FS_TICK     = 13   # tick labels assi
FS_ANNOT    = 12   # numeri sopra le barre
FS_LEGEND   = 12   # legenda


#  Present class in dataset
present_directions = [c for c in DIRECTION_CLASSES if c in df['direction'].values]
present_sides      = [c for c in SIDE_CLASSES      if c in df['side'].values]
present_boats      = [c for c in BOAT_TYPE_CLASSES  if c in df['boat_type'].values]

#  Graph: Class distribution — full dataset

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Class Distribution — Full Dataset', fontweight='bold', fontsize=FS_TITLE)

dir_counts  = df['direction'].value_counts().reindex(present_directions, fill_value=0)
side_counts = df['side'].value_counts().reindex(present_sides,           fill_value=0)
boat_counts = df['boat_type'].value_counts().reindex(present_boats,      fill_value=0)

for ax, counts, color, title, rot in [
    (axes[0], dir_counts,  'steelblue',  'Direction', 30),
    (axes[1], side_counts, 'seagreen',   'Side',       0),
    (axes[2], boat_counts, 'darkorange', 'Boat Type', 30),
]:
    ax.bar(counts.index, counts.values, color=color, edgecolor='white')
    ax.set_title(title, fontsize=FS_SUBTITLE)
    ax.tick_params(axis='x', rotation=rot, labelsize=FS_TICK)
    ax.tick_params(axis='y', labelsize=FS_TICK)
    # Margine superiore per evitare sovrapposizioni
    ax.set_ylim(0, counts.values.max() * 1.15)
    offset = ax.get_ylim()[1] * 0.015
    for bar, val in zip(ax.patches, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + offset,
                str(val), ha='center', va='bottom', fontsize=FS_ANNOT)

plt.tight_layout()
plt.savefig(DATA_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Graph 2: Train / Val / Test for each class
split_colors = {'train': 'steelblue', 'val': 'seagreen', 'test': 'darkorange'}
splits       = {'train': df_train, 'val': df_val, 'test': df_test}
tasks = [
    ('Direction', 'direction', present_directions),
    ('Side',      'side',      present_sides),
    ('Boat Type', 'boat_type', present_boats),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Train / Val / Test Split per Class', fontweight='bold', fontsize=FS_TITLE)
width = 0.25

for ax, (title, col, present_classes) in zip(axes, tasks):
    x = np.arange(len(present_classes))
    all_vals = []
    for i, (split_name, split_df) in enumerate(splits.items()):
        counts = (split_df[col]
                  .value_counts()
                  .reindex(present_classes, fill_value=0))
        all_vals.extend(counts.values)
        bars = ax.bar(x + i * width, counts.values,
                      width=width,
                      label=split_name,
                      color=split_colors[split_name],
                      edgecolor='white')
        for bar, val in zip(bars, counts.values):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 2,
                        str(val), ha='center', va='bottom',
                        fontsize=FS_ANNOT - 1,   # leggermente più piccolo
                        )              # verticale → non si sovrappone
    # Margine superiore dinamico
    ax.set_ylim(0, max(all_vals) * 1.20)
    ax.set_title(title, fontsize=FS_SUBTITLE)
    ax.set_xticks(x + width)
    ax.set_xticklabels(present_classes, rotation=30, ha='right', fontsize=FS_TICK)
    ax.tick_params(axis='y', labelsize=FS_TICK)
    ax.legend(fontsize=FS_LEGEND)

plt.tight_layout()
plt.savefig(DATA_DIR / 'split_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Temporal Statistics

In [ ]:
# Per-session summary: frame count, duration, labels
session_stats = (
    df.groupby('session')
      .agg(
          n_frames   = ('frame_number', 'count'),
          duration_s = ('timestamp_s',  'max'),
          direction  = ('direction',    'first'),
          side       = ('side',         'first'),
          boat_type  = ('boat_type',    'first'),
      )
      .reset_index()
)

print(f'Session statistics ({len(session_stats)} sessions):')
print(session_stats[['session', 'n_frames', 'duration_s', 'direction', 'side', 'boat_type']]
      .to_string(index=False))

print(f'\nFrame count  - mean: {session_stats.n_frames.mean():.0f}'
      f'  min: {session_stats.n_frames.min()}'
      f'  max: {session_stats.n_frames.max()}')
print(f'Duration (s) - mean: {session_stats.duration_s.mean():.1f}'
      f'  min: {session_stats.duration_s.min():.1f}'
      f'  max: {session_stats.duration_s.max():.1f}')


In [ ]:
# Histograms of session frame counts and durations
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Session Temporal Statistics', fontweight='bold')

median_frames = session_stats['n_frames'].median()
axes[0].hist(session_stats['n_frames'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Frames per session')
axes[0].set_ylabel('Count')
axes[0].set_title('Frame count distribution')
axes[0].axvline(median_frames, color='red', ls='--', label=f'median={median_frames:.0f}')
axes[0].legend()

median_dur = session_stats['duration_s'].median()
axes[1].hist(session_stats['duration_s'], bins=20, color='seagreen', edgecolor='white')
axes[1].set_xlabel('Duration (seconds)')
axes[1].set_ylabel('Count')
axes[1].set_title('Video duration distribution')
axes[1].axvline(median_dur, color='red', ls='--', label=f'median={median_dur:.1f}s')
axes[1].legend()

plt.tight_layout()
plt.savefig(DATA_DIR / 'session_duration_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Duration distribution plot saved')


## Done

| Output file | Description |
|---|---|
| `data/train.csv` | Training manifest |
| `data/val.csv` | Validation manifest |
| `data/test.csv` | Test manifest |
| `data/class_info.json` | Class names, index mappings, weights, FPS values, temporal params |
| `data/class_distribution.png` | Class balance chart |
| `data/session_duration_distribution.png` | Session length histograms |
